# Audio Augmentation Demo

This notebook demonstrates audio augmentation techniques:
1. **Time Domain**: Noise, time stretch, time shift
2. **Frequency Domain**: Pitch shift
3. **Spectrogram**: SpecAugment (time/frequency masking)

**Key Principle**: Augmentation must preserve the label (e.g., same word, same speaker intent)!

In [ ]:
# Install required packages
# !pip install audiomentations librosa soundfile matplotlib numpy torchaudio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

## 1. Create Sample Audio

We'll create a simple synthetic audio signal for demonstration.

In [ ]:
# Create a sample audio: a simple tone with harmonics (simulating speech-like signal)
sr = 16000  # Sample rate (common for speech)
duration = 2.0  # 2 seconds
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Create a more interesting signal (multiple frequencies like speech)
f1, f2, f3 = 200, 400, 600  # Fundamental + harmonics
audio = 0.5 * np.sin(2 * np.pi * f1 * t)
audio += 0.3 * np.sin(2 * np.pi * f2 * t)
audio += 0.2 * np.sin(2 * np.pi * f3 * t)

# Add envelope to make it more speech-like
envelope = np.exp(-t * 0.5) * (1 - np.exp(-t * 10))
audio = audio * envelope
audio = audio.astype(np.float32)

print(f"Sample rate: {sr} Hz")
print(f"Duration: {duration} seconds")
print(f"Samples: {len(audio)}")

print("\n🔊 Original Audio:")
display(Audio(audio, rate=sr))

In [ ]:
# Visualization helper
def plot_audio_comparison(original, augmented, sr, title, aug_name):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Waveforms
    times_orig = np.arange(len(original)) / sr
    times_aug = np.arange(len(augmented)) / sr
    
    axes[0, 0].plot(times_orig, original, color='blue', alpha=0.7)
    axes[0, 0].set_title('Original Waveform', fontsize=12)
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('Amplitude')
    
    axes[0, 1].plot(times_aug, augmented, color='green', alpha=0.7)
    axes[0, 1].set_title(f'{aug_name} Waveform', fontsize=12)
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Amplitude')
    
    # Spectrograms
    D_orig = librosa.amplitude_to_db(np.abs(librosa.stft(original)), ref=np.max)
    D_aug = librosa.amplitude_to_db(np.abs(librosa.stft(augmented)), ref=np.max)
    
    librosa.display.specshow(D_orig, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 0])
    axes[1, 0].set_title('Original Spectrogram', fontsize=12)
    
    librosa.display.specshow(D_aug, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 1])
    axes[1, 1].set_title(f'{aug_name} Spectrogram', fontsize=12)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 2. Time Domain Augmentations

In [ ]:
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift

# 2.1 Add Gaussian Noise
noise_aug = AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.015, p=1.0)
audio_noisy = noise_aug(samples=audio.copy(), sample_rate=sr)

print("🔊 With Gaussian Noise:")
display(Audio(audio_noisy, rate=sr))

plot_audio_comparison(audio, audio_noisy, sr, 
                      'Adding Gaussian Noise (simulates recording conditions)', 
                      'Noisy')

In [ ]:
# 2.2 Time Stretching (make faster/slower without pitch change)
stretch_aug = TimeStretch(min_rate=0.8, max_rate=0.8, p=1.0)  # Slower
audio_slow = stretch_aug(samples=audio.copy(), sample_rate=sr)

stretch_aug_fast = TimeStretch(min_rate=1.2, max_rate=1.2, p=1.0)  # Faster
audio_fast = stretch_aug_fast(samples=audio.copy(), sample_rate=sr)

print("🔊 Time Stretch (0.8x - Slower):")
display(Audio(audio_slow, rate=sr))

print("\n🔊 Time Stretch (1.2x - Faster):")
display(Audio(audio_fast, rate=sr))

plot_audio_comparison(audio, audio_slow, sr,
                      'Time Stretching: Same pitch, different duration',
                      'Stretched (0.8x)')

In [ ]:
# 2.3 Time Shift
shift_aug = Shift(min_shift=-0.3, max_shift=0.3, p=1.0)
audio_shifted = shift_aug(samples=audio.copy(), sample_rate=sr)

print("🔊 Time Shifted:")
display(Audio(audio_shifted, rate=sr))

plot_audio_comparison(audio, audio_shifted, sr,
                      'Time Shift: Audio starts at different point',
                      'Shifted')

## 3. Frequency Domain Augmentations

In [ ]:
# 3.1 Pitch Shift
pitch_up = PitchShift(min_semitones=3, max_semitones=3, p=1.0)
pitch_down = PitchShift(min_semitones=-3, max_semitones=-3, p=1.0)

audio_pitch_up = pitch_up(samples=audio.copy(), sample_rate=sr)
audio_pitch_down = pitch_down(samples=audio.copy(), sample_rate=sr)

print("🔊 Pitch Shifted UP (+3 semitones):")
display(Audio(audio_pitch_up, rate=sr))

print("\n🔊 Pitch Shifted DOWN (-3 semitones):")
display(Audio(audio_pitch_down, rate=sr))

plot_audio_comparison(audio, audio_pitch_up, sr,
                      'Pitch Shift: Higher frequency, same duration',
                      'Pitch +3')

## 4. SpecAugment: Spectrogram Masking

Used by Google's speech recognition - mask time and frequency bands.

In [ ]:
import torch
import torchaudio.transforms as T

# Compute spectrogram
spec_transform = T.Spectrogram(n_fft=512, hop_length=128)
audio_tensor = torch.tensor(audio).unsqueeze(0)
spectrogram = spec_transform(audio_tensor)

# SpecAugment: Time and Frequency Masking
freq_mask = T.FrequencyMasking(freq_mask_param=30)
time_mask = T.TimeMasking(time_mask_param=40)

# Apply masking
spec_freq_masked = freq_mask(spectrogram.clone())
spec_time_masked = time_mask(spectrogram.clone())
spec_both_masked = time_mask(freq_mask(spectrogram.clone()))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def plot_spec(ax, spec, title):
    im = ax.imshow(spec.squeeze().log2().numpy(), aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Frequency Bin')
    ax.set_xlabel('Time Frame')
    return im

plot_spec(axes[0, 0], spectrogram, 'Original Spectrogram')
plot_spec(axes[0, 1], spec_freq_masked, 'Frequency Masking')
plot_spec(axes[1, 0], spec_time_masked, 'Time Masking')
plot_spec(axes[1, 1], spec_both_masked, 'SpecAugment (Both)')

plt.suptitle('SpecAugment: Masking for Robust Speech Recognition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Combined Augmentation Pipeline

In [ ]:
# Create a combined pipeline
augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.5),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    Shift(min_shift=-0.1, max_shift=0.1, p=0.5),
])

print("🎲 Random Augmentation Pipeline (5 different augmentations):")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Original
axes[0, 0].plot(np.arange(len(audio)) / sr, audio, color='blue')
axes[0, 0].set_title('Original', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Time (s)')

print("\n🔊 Original:")
display(Audio(audio, rate=sr))

for i in range(5):
    np.random.seed(i * 42)  # Different augmentation each time
    augmented = augment_pipeline(samples=audio.copy(), sample_rate=sr)
    
    ax = axes.flat[i + 1]
    ax.plot(np.arange(len(augmented)) / sr, augmented, color='green', alpha=0.8)
    ax.set_title(f'Augmentation {i+1}', fontsize=11)
    ax.set_xlabel('Time (s)')
    
    print(f"\n🔊 Augmentation {i+1}:")
    display(Audio(augmented, rate=sr))

plt.suptitle('Same Audio → 5 Different Training Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ⚠️ Dangerous Augmentations

Some augmentations can change the meaning/label!

In [ ]:
print("⚠️ DANGEROUS Augmentations by Task:")
print("=" * 60)
print()
print("📢 Speech Recognition:")
print("  ✅ Safe: Noise, reverb, mild time stretch")
print("  ❌ Dangerous: Heavy pitch shift (distorts phonemes)")
print()
print("👤 Speaker Identification:")
print("  ✅ Safe: Noise, reverb")
print("  ❌ Dangerous: Pitch shift (CHANGES THE VOICE!)")
print()
print("🎵 Music Genre Classification:")
print("  ✅ Safe: Time stretch, noise")
print("  ❌ Dangerous: Pitch shift (changes the musical key!)")
print()
print("😊 Emotion Recognition:")
print("  ✅ Safe: Light noise, reverb")
print("  ❌ Dangerous: Speed change (fast → excited, slow → sad)")

In [ ]:
# Demonstrate: Extreme pitch shift changes voice identity
extreme_pitch_up = PitchShift(min_semitones=8, max_semitones=8, p=1.0)
audio_high = extreme_pitch_up(samples=audio.copy(), sample_rate=sr)

print("Example: Extreme Pitch Shift")
print("\n🔊 Original (Speaker A):")
display(Audio(audio, rate=sr))

print("\n🔊 +8 Semitones (Sounds like different speaker!):")
display(Audio(audio_high, rate=sr))
print("\n❌ For speaker ID, this would be WRONG to use!")

## Key Takeaways

1. **Time domain** augmentations (noise, stretch, shift) are generally safe
2. **Pitch shift** can be dangerous for speaker/emotion tasks
3. **SpecAugment** is very effective for speech recognition
4. **Always consider what defines your label** before choosing augmentations
5. **Combine multiple mild augmentations** rather than one extreme one